In [2]:
import sys, os, io
os.environ["KALEDO_USE_SWIFT"] = "true"

import numpy as np
#import matplotlib.pyplot as plt
from SpaceBalls.utils import get_two_perp_unit_vectors
from SpaceBalls.sph_meshing import get_mesh_from_fibonacci_sphere, get_edge_connectivities
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

import kaleido
import plotly
import plotly.graph_objects as go
import imageio
from PIL import Image
from SpaceBalls.paths import MEDIA_DIR


def export_sphere_rotation_gif(fig, filename="rotated_sphere.gif", num_frames=36, fps=15):
    """
    Rotates the camera horizontally around a Plotly 3D figure and saves it as a GIF.
    
    Parameters:
    - fig: plotly.graph_objects.Figure object
    - filename: str, path/name of the output GIF file
    - num_frames: int, total frames for a full 360-degree rotation (higher = smoother)
    - fps: int, frames per second for the output GIF
    """
    # 1. Safely extract current camera position or fall back to defaults
    try:
        eye = fig.layout.scene.camera.eye
        x_init = eye.x if eye.x is not None else 1.25
        y_init = eye.y if eye.y is not None else 1.25
        z_fixed = eye.z if eye.z is not None else 1.25
    except AttributeError:
        # Default Plotly 3D camera eye values if untouched
        x_init, y_init, z_fixed = 1.25, 1.25, 1.25

    # 2. Calculate radius in the XY plane and the starting angle
    radius = np.sqrt(x_init**2 + y_init**2)
    start_angle = np.arctan2(y_init, x_init)
    
    # Calculate duration per frame in milliseconds
    frame_duration = int(1000 / fps)
    
    # Create a deep copy so we don't accidentally mutate your original figure
    fig_working = go.Figure(fig)
    
    frames = []
    
    print("Rendering frames... Grab a coffee, this takes a few seconds.")
    
    # 3. Orbit the camera and capture frames
    for i in range(num_frames):
        # Calculate current angle for this frame
        angle = start_angle + (2 * np.pi * i / num_frames)
        
        # Compute new camera coordinates (X and Y rotate, Z stays level)
        new_x = radius * np.cos(angle)
        new_y = radius * np.sin(angle)
        
        # Update the camera position
        fig_working.update_layout(
            scene_camera=dict(
                eye=dict(x=new_x, y=new_y, z=z_fixed)
            )
        )
        
        # Render the figure to raw PNG bytes in memory
        img_bytes = fig_working.to_image(format="png")
        
        # Append the PIL Image object to our frame container
        frames.append(Image.open(io.BytesIO(img_bytes)))

    # 4. Compile and save the GIF
    frames[0].save(
        filename,
        save_all=True,
        append_images=frames[1:],
        duration=frame_duration,
        loop=0, # 0 means infinite loop
        optimize=True
    )
    
    print(f"Success! Animation successfully saved to {filename}")
    
def add_scatter(fig, points, color_values=None):
    fig.add_trace(go.Scatter3d(
    x=points[:, 0],
    y=points[:, 1],
    z=points[:, 2],
    mode='markers',
    marker=dict(
        size=3,
        color=color_values,
        colorscale='plasma',
        #showscale=True,
        #colorbar=dict(title='Weights')
    ),
    #name='Grid Points'
))
    
def add_facets(fig, vertices, i, j, k):
    fig.add_trace(go.Mesh3d(
    x=vertices[:, 0],
    y=vertices[:, 1],
    z=vertices[:, 2],
    i=i,
    j=j,
    k=k,
    opacity=0.85,
    color='white',
    flatshading=True,
    name='Hull triangles',
    lighting=dict(
            ambient=0.9,
            diffuse=0.5,
            specular=0.1,
            roughness=0.9,  # Low roughness for a glossy reflection
            fresnel=0.9
        ),
))

def get_edges_to_plot(hull):

    edge_conns = get_edge_connectivities(hull)

    triangle_vertices = hull.points[hull.simplices]
    triangle_centroids = np.mean(triangle_vertices, axis=1)

    edge_x = []
    edge_y = []
    edge_z = []
    for i, j in edge_conns:
        p0 = hull.points[i]
        p1 = hull.points[j]
        edge_x += [p0[0], p1[0], None]
        edge_y += [p0[1], p1[1], None]
        edge_z += [p0[2], p1[2], None]
    
    return edge_x, edge_y, edge_z




In [3]:
from SpaceBalls.sph_meshing import QuadratureGrid, get_edge_connectivities, get_faceted_sphere_mesh
from config.constants import earth_radius
import scipy.spatial
RE = earth_radius()

radius_meters = 0.5
n_faces = 48
default_golden = False
mesh = get_faceted_sphere_mesh(radius_meters, n_faces, default_golden=default_golden)
vertices, centroids, normal_vecs, areas, edge_conns, hull = mesh


# Create figure
fig = go.Figure()
add_scatter(fig, vertices)
add_scatter(fig, centroids, color_values=areas)
#add_scatter(fig, normal_vecs)

i, j, k = hull.simplices.T
add_facets(fig, vertices, i, j, k)


edge_x, edge_y, edge_z = get_edges_to_plot(hull)
fig.add_trace(go.Scatter3d(
    x=edge_x,
    y=edge_y,
    z=edge_z,
    mode='lines',
    line=dict(color='black', width=2),
    #name='edge_conns'
))

axis_lim = radius_meters 
fig.update_layout(
    scene=dict(
        xaxis=dict(visible=False, range=[-axis_lim, axis_lim]),
        yaxis=dict(visible=False, range=[-axis_lim, axis_lim]),
        zaxis=dict(visible=False, range=[-axis_lim, axis_lim]),
        aspectmode='cube',
        camera=dict(
            eye=dict(x=1.0, y=1.0, z=0.3)  # smaller values => zoom in
        ),
    ),
    width=500,
    height=500,
    showlegend=False,
    margin=dict(l=20, r=20, t=20, b=20),  # Padding in pixels
    #pad=0   
)


fig.show()
name_tail = '_golden' if default_golden else ''

#fig.write_image(os.path.join(MEDIA_DIR, 'figures', 'sc_shapes',
#                             'sphere'+str(n_faces)+name_tail),
#                             format='png')


"""
fig, ax = plot_sc_shape(hull.points, stacked_areas=None, radius=1, displace_points=True)
for idx_triad in hull.simplices:
    triangle = hull.points[idx_triad]
    ax.plot(triangle[:,0], triangle[:,1], triangle[:,2])
"""

Number of faces: 48
Number of faces: 48


'\nfig, ax = plot_sc_shape(hull.points, stacked_areas=None, radius=1, displace_points=True)\nfor idx_triad in hull.simplices:\n    triangle = hull.points[idx_triad]\n    ax.plot(triangle[:,0], triangle[:,1], triangle[:,2])\n'

In [6]:

export_sphere_rotation_gif(
    fig=fig, 
    filename="my_rotating_sphere.gif", 
    num_frames=45,  # 45 frames gives a very fluid rotation
    fps=20          # Plays back at 20 frames per second
)

Rendering frames... Grab a coffee, this takes a few seconds.


KaleidoError: Error 525: gl-shader: Error linking program: 

In [ ]:
def plot_sc_shape(stacked_normals, stacked_areas=None, radius=None, plot_surfaces=False, displace_points=True):

    all_points = stacked_normals * radius

    fig = plt.figure()
    ax = fig.add_subplot(projection='3d')

    if displace_points:
        ax.scatter(all_points[:,0], all_points[:,1], all_points[:,2], color='r', marker='.')

    if plot_surfaces:
        # Plot square surfaces
        for point, normal, area in zip(all_points, stacked_normals, stacked_areas):
            square = create_square_surface(point, normal, area)
            square_poly = Poly3DCollection([square], alpha=0.5, color='cyan')
            ax.add_collection3d(square_poly)
    
    ax.set_box_aspect([1, 1, 1])
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')

    return fig, ax



def create_square_surface(center, normal, area):
    # Normalize normal
    normal = normal / np.linalg.norm(normal)
    
    # Find two vectors orthogonal to the normal
    u, v = get_two_perp_unit_vectors(normal)
    
    side = np.sqrt(area)
    u *= side / 2
    v *= side / 2

    # Four corners of the square
    corners = [
        center + u + v,
        center + u - v,
        center - u - v,
        center - u + v
    ]
    return corners